
# RNN vs LSTM vs GRU

## Instructions

In this lab, you will build **three text classification models** from scratch:
- RNN
- LSTM
- GRU

---

### Objectives
By the end of this lab, you should be able to:

- Preprocess text data
- Build a vocabulary
- Encode and pad sequences
- Implement RNN, LSTM, and GRU in PyTorch
- Train and evaluate models 
- Compare architectures


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from datasets import load_dataset

from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
label_map = {
    "Human": 0,
    "Anthropic": 1,
    "Google": 2,
    "OpenAI": 3,
    "Meta": 4
}

In [4]:
# data loading
dataset = load_dataset("csv", data_files="../dataset_final.csv", sep=";")

dataset = dataset["train"].train_test_split(test_size=0.2)

train_data = list(dataset["train"])
test_data = list(dataset["test"])

# remove exemples without labels
train_data = [x for x in train_data if x["Label"] is not None]
test_data = [x for x in test_data if x["Label"] is not None]

print("Train size:", len(train_data))
print("Test size:", len(test_data))

Generating train split: 0 examples [00:00, ? examples/s]

Train size: 5596
Test size: 1400


In [5]:
print(dataset["train"].features)
print(train_data[0])

{'ID': Value('string'), 'Text': Value('string'), 'Label': Value('string')}
{'ID': 'D1-1419', 'Text': 'We present the structure of the fully relaxed (001) surface of the half-metallic manganite La0.7Sr0.3MnO3, calculated using density functional theory within the generalized gradient approximation (GGA). Two relevant ferroelastic order parameters are identified and characterized: The tilting of the oxygen octahedra, which is present in the bulk phase, oscillates and decreases towards the surface, and an additional ferrodistortive Mn off-centering, triggered by the surface, decays monotonically into the bulk. The narrow d-like energy band that is characteristic of unrelaxed manganite surfaces is shifted down in energy by these structural distortions, retaining its uppermost layer localization. The magnitude of the zero-temperature magnetization is unchanged from its bulk value, but the effective spin-spin interactions are reduced at the surface.', 'Label': 'Human'}


In [6]:
sum(1 for x in train_data if x["Label"] is None)

0

In [7]:
train_texts = [x["Text"] for x in train_data]
test_texts = [x["Text"] for x in test_data]

train_labels = [label_map[x["Label"].strip()] for x in train_data]
test_labels = [label_map[x["Label"].strip()] for x in test_data]


# Part 1 – Text Preprocessing

You must:

1. Write a `tokenize(text)` function.
2. Build a vocabulary using the training set only.
3. Keep only the top 10,000 most frequent words.
4. Add special tokens:
   - `<pad>`
   - `<unk>`
5. Explain in a markdown cell:
   - Why do we not build the vocabulary using the test set?


In [8]:
import re

def tokenize(text):
    text = str(text)
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    return text.split()

In [9]:
tokenize(train_data)

['id',
 'd',
 'text',
 'we',
 'present',
 'the',
 'structure',
 'of',
 'the',
 'fully',
 'relaxed',
 'surface',
 'of',
 'the',
 'half',
 'metallic',
 'manganite',
 'la',
 'sr',
 'mno',
 'calculated',
 'using',
 'density',
 'functional',
 'theory',
 'within',
 'the',
 'generalized',
 'gradient',
 'approximation',
 'gga',
 'two',
 'relevant',
 'ferroelastic',
 'order',
 'parameters',
 'are',
 'identified',
 'and',
 'characterized',
 'the',
 'tilting',
 'of',
 'the',
 'oxygen',
 'octahedra',
 'which',
 'is',
 'present',
 'in',
 'the',
 'bulk',
 'phase',
 'oscillates',
 'and',
 'decreases',
 'towards',
 'the',
 'surface',
 'and',
 'an',
 'additional',
 'ferrodistortive',
 'mn',
 'off',
 'centering',
 'triggered',
 'by',
 'the',
 'surface',
 'decays',
 'monotonically',
 'into',
 'the',
 'bulk',
 'the',
 'narrow',
 'd',
 'like',
 'energy',
 'band',
 'that',
 'is',
 'characteristic',
 'of',
 'unrelaxed',
 'manganite',
 'surfaces',
 'is',
 'shifted',
 'down',
 'in',
 'energy',
 'by',
 'these',

In [10]:
from collections import Counter

def build_vocab(texts, max_words=10000):

    counter = Counter()

    for text in texts:

        if text is None:
            continue

        tokens = tokenize(text)
        counter.update(tokens)

    most_common = counter.most_common(max_words)

    vocab = {
        "<pad>": 0,
        "<unk>": 1
    }

    for word, _ in most_common:
        vocab[word] = len(vocab)

    return vocab

In [11]:
vocab = build_vocab(train_texts)


# Part 2 – Encoding and Padding

You must:

1. Create an `encode(text)` function.
2. Convert tokens into vocabulary indices.
3. Pad or truncate sequences to a fixed length (e.g., 25).
4. Create a custom `collate()` function.
5. Create train, validation, and test DataLoaders.

Explain:
- Why is padding necessary?
- Why should validation and test loaders not shuffle?


In [12]:
def encode(vocab, text, max_len=25):
    tokens = tokenize(text)

    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]

    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids

In [13]:
def one_hot_encode(vocab, token):
    vector = np.zeros(len(vocab))
    
    token_index = vocab.get(token, vocab["<unk>"])
    vector[token_index] = 1
    
    return vector

In [14]:
train_loader = DataLoader(
    dataset["train"],
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    dataset["test"],
    batch_size=32,
    shuffle=False
)


# Part 3 – Model Implementation

Create a class called `Model` that:

- Uses an Embedding layer
- Supports:
  - RNN
  - LSTM
  - GRU
- Uses multiple layers
- Applies dropout
- Outputs class logits

Your model must accept:
- model_type
- vocab_size
- embed_dim
- hidden_dim
- num_layers

Explain:
- The internal difference between RNN, LSTM, and GRU.


In [15]:
class Model(nn.Module):
    def __init__(self, model_type, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)

        if model_type == "RNN":
            self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers, batch_first=True)
        elif model_type == "LSTM":
            self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        else:
            self.rnn = nn.GRU(embed_dim, hidden_dim, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 4)

        def forward(self, x):
            x = self.embedding(x)
            x = self.drpout(x)
            out, h = self.rnn(x)
            if isinstance(h, tuple):
                h = h[0]
                return self.fc(h[-1])


# Part 4 – Training Loop

Implement:

- A full training loop
- Validation loop
- Accuracy computation
- Loss tracking per epoch

Train for 10-50 epochs.

Store:
- train_loss
- val_loss
- train_accuracy
- val_accuracy

Explain:
- Why do we use `model.train()` and `model.eval()`?


In [16]:
def train(model, train_loader, val_loader, criterion, epochs = 10, lr = 0.001, verbose = True):
    ## verbose - print losses and accuracies per epoch
    
    train_accs = []
    val_accs = []
    train_losses = []
    val_losses = []
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    for epoch in range(epochs): 
        model.train()
        for x, y in train_loader:
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        train_loss, train_acc = evaluate(model, train_loader, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        
        train_accs.append(train_acc)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        val_losses.append(val_loss)

        if verbose: 
            print(f"Epoch {epoch+1}")
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
            print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    
    return train_accs, val_accs, train_losses, val_losses


# Part 5 – Model Comparison

Train:
- RNN
- LSTM
- GRU

Track validation accuracy and determine:
- Which performs best?
- Why?

Plot:
- Loss curves
- Accuracy curves

Explain signs of overfitting.


In [17]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            outputs = model(x)
            loss = criterion(outputs, y)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / len(loader), correct / total

In [18]:
def plot_values(list_values, list_labels  = ["Train", "Validation"], ylabel = "Accuracy", title = None):
    plt.figure()
    for i in range(len(list_values)):
        plt.plot(list_values[i], label = list_labels[i])
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.legend()
    if title is not None: plt.title()
    plt.show()

In [19]:
def load_dataset_embed(filespath, max_words, max_len=100, batch_size=32):

    dataset = load_dataset("csv", data_files=filespath,sep=";")

    dataset = dataset["train"].train_test_split(test_size=0.2)

    train_data = dataset["train"]
    test_data = dataset["test"]

    train_texts = [x["Text"] for x in train_data]
    train_labels = [x["Label"] for x in train_data]

    test_texts = [x["Text"] for x in test_data]
    test_labels = [x["Label"] for x in test_data]

    vocab = build_vocab(train_texts, max_words)

    def collate(batch):
        texts = [x["Text"] for x in batch if x["Label"] is not None]
        labels = [
            label_map[x["Label"].strip()]
            for x in batch
            if x["Label"] is not None
        ]
    
        encoded = [encode(vocab, t, max_len) for t in texts]
    
        x = torch.tensor(encoded, dtype=torch.long)
        y = torch.tensor(labels, dtype=torch.long)
        return x.to(device), y.to(device)

    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, collate_fn=collate)

    val_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=collate)

    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, collate_fn=collate)

    return train_loader, val_loader, test_loader, vocab

In [20]:
class RNNClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 5)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        last_hidden = hidden[-1]
        out = self.fc(last_hidden)
        return out

In [21]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.0, bidirectional=False, num_classes=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        direction_factor = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * direction_factor, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)     # (B, L, E)
        output, (hidden, cell) = self.lstm(embedded)

        if self.lstm.bidirectional:
            forward_hidden = hidden[-2]
            backward_hidden = hidden[-1]
            last_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        else:
            last_hidden = hidden[-1]

        last_hidden = self.dropout(last_hidden)

        out = self.fc(last_hidden)

        return out   # (B, num_classes)

In [22]:
class GRUClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.0, bidirectional=False, num_classes=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)
        direction_factor = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * direction_factor, num_classes)

    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)

        if self.gru.bidirectional:
            forward_hidden = hidden[-2]
            backward_hidden = hidden[-1]
            last_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        else:
            last_hidden = hidden[-1]
        last_hidden = self.dropout(last_hidden)
        out = self.fc(last_hidden)
        return out   # (B, num_classes)

In [23]:
def test_rnn():
    filespath = "../dataset_final.csv"
    max_words = 20000
    max_len = 100
    embed_dim = 200
    train_loader, val_loader, test_loader, vocab_local = load_dataset_embed(filespath, max_words, max_len = max_len)
    vocab_size_real = len(vocab_local)
    model = RNNClassifier(vocab_size_real,embed_dim, hidden_dim=128, num_layers=1)
    criterion = nn.CrossEntropyLoss()
    train_accs, val_accs, train_losses, val_losses = train(model, train_loader, val_loader, criterion, epochs = 5)

    plot_values([train_accs, val_accs])
    plot_values([train_losses, val_losses], ylabel = "Loss")
    
    _, test_acc = evaluate(model, test_loader, criterion)
    print(f"Test Accuracy: {test_acc:.4f}")
    confusion_matrix_eval(model, test_loader)
    return model, vocab_local

def test_lstm():
    filespath = "../dataset_final.csv"
    max_words = 20000
    max_len = 100 ## do not increase
    embed_dim = 200
    train_loader, val_loader, test_loader, vocab_local = load_dataset_embed(filespath, max_words, max_len = max_len)
    vocab_size_real = len(vocab_local)
    model = LSTMClassifier(vocab_size_real,embed_dim, hidden_dim=128, num_layers=2, bidirectional = True, dropout = 0.5)
    criterion = nn.CrossEntropyLoss()
    train_accs, val_accs, train_losses, val_losses = train(model, train_loader, val_loader, criterion, epochs = 5) #, lr = 0.0001)

    plot_values([train_accs, val_accs])
    plot_values([train_losses, val_losses], ylabel = "Loss")
    
    _, test_acc = evaluate(model, test_loader, criterion)
    print(f"Test Accuracy: {test_acc:.4f}")
    confusion_matrix_eval(model, test_loader)
    return model, vocab_local

def test_gru():
    filespath = "../dataset_final.csv"
    max_words = 20000
    max_len = 100 ## do not increase
    embed_dim = 200
    train_loader, val_loader, test_loader, vocab_local = load_dataset_embed(filespath, max_words, max_len = max_len)
    vocab_size_real = len(vocab_local)
    model = GRUClassifier(vocab_size_real,embed_dim, hidden_dim=128, num_layers=1, bidirectional = True, dropout = 0.3)
    criterion = nn.CrossEntropyLoss()
    train_accs, val_accs, train_losses, val_losses = train(model, train_loader, val_loader, criterion, epochs = 5) #, lr = 0.0001)
    
    plot_values([train_accs, val_accs])
    plot_values([train_losses, val_losses], ylabel = "Loss")
    
    _, test_acc = evaluate(model, test_loader, criterion)
    print(f"Test Accuracy: {test_acc:.4f}")
    confusion_matrix_eval(model, test_loader)
    return model, vocab_local


# Part 6 – Final Evaluation

Using the best model:

1. Evaluate on the test set.
2. Compute test accuracy.
3. Plot a confusion matrix.

Explain:
- Which classes are most confused?
- Why might that happen?


In [24]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def confusion_matrix_eval(model, loader):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:

            outputs = model(x)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)

    disp = ConfusionMatrixDisplay(cm)
    disp.plot()

In [ ]:
#model_rnn, vocab_do_treino = test_rnn()
#model_lstm, vocab_do_treino = test_lstm()
model_gru, vocab_do_treino = test_gru()

Epoch 1
Train Loss: 0.8022, Train Acc: 0.6973
Val   Loss: 0.9856, Val   Acc: 0.5936
Epoch 2
Train Loss: 0.4603, Train Acc: 0.8349
Val   Loss: 0.8567, Val   Acc: 0.6600



# Part 7 - submission file


In [39]:
def fill_csv(model):
    dataset = load_dataset("csv", data_files="../subm1.csv",sep=";")["train"]
    texts = dataset["Text"]
    max_len=100
    predictions = []

    inverse_label_map = {v: k for k, v in label_map.items()}

    model.eval()
    with torch.no_grad():
        for text in texts:
            seq = encode(vocab, text, max_len)
            x = torch.tensor(seq).unsqueeze(0).to(device)

            outputs = model(x)
            pred_idx = torch.argmax(outputs, dim=1).item()
            pred_label = inverse_label_map[pred_idx]

            predictions.append(pred_label)

    dataset = dataset.add_column("Label", predictions)

    # save CSV
    with open("../subm1-g6-MEI-B.csv", "w", encoding="utf8") as f:
        cols = dataset.column_names
        f.write(";".join(cols) + "\n")
        for row in dataset:
            f.write(";".join(str(row[c]) for c in cols) + "\n")

In [40]:
fill_csv(model_lstm)